# HW04-20234080115-李其隆

## 2 序列模型

### 2.1 理论计算题

字符序列为 `ababc`，相邻字符转移为 `a->b`, `b->a`, `a->b`, `b->c`。在给定前一个字符为 `b` 时，出现过的转移为 `b->a` 和 `b->c`，因此`count(b)=2`, `count(b,a)=1`, `count(b,c)=1`。

词汇表大小为 $|V|=3$，使用拉普拉斯平滑：

$$
p(x_t=j \mid x_{t-1}=i)=\frac{count(i,j)+1}{count(i)+|V|}
$$

所以：

$$
p('a'\mid 'b')=\frac{1+1}{2+3}=\frac{2}{5}=0.4
$$

$$
p('c'\mid 'b')=\frac{1+1}{2+3}=\frac{2}{5}=0.4
$$

In [1]:
from collections import Counter

sequence = "ababc"
vocab = ["a", "b", "c"]
transitions = list(zip(sequence[:-1], sequence[1:]))
transition_counts = Counter(transitions)
prev_counts = Counter(sequence[:-1])

def laplace_transition_prob(prev_char, next_char):
    numerator = transition_counts[(prev_char, next_char)] + 1
    denominator = prev_counts[prev_char] + len(vocab)
    return numerator / denominator

print("Transitions:", transitions)
print("p('a' | 'b') =", laplace_transition_prob("b", "a"))
print("p('c' | 'b') =", laplace_transition_prob("b", "c"))

Transitions: [('a', 'b'), ('b', 'a'), ('a', 'b'), ('b', 'c')]
p('a' | 'b') = 0.4
p('c' | 'b') = 0.4


### 2.2 编程题

实现 `preprocess_text(text, n)`：将文本转小写、去除标点符号并保留字母和空格，然后按空格分词，按词频构建词汇表，最后用长度为 `n` 的滑动窗口生成特征序列和下一个词标签。如果窗口后没有后续词，则标签为 `None`。

In [2]:
import re
from collections import Counter

def preprocess_text(text, n):
    if n <= 0:
        raise ValueError("n must be a positive integer")

    cleaned = re.sub(r"[^a-z\s]", " ", text.lower())
    tokens = cleaned.split()

    frequencies = Counter(tokens)
    first_position = {word: i for i, word in enumerate(tokens) if word not in tokens[:i]}
    sorted_words = sorted(frequencies, key=lambda w: (-frequencies[w], first_position[w]))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    features, labels = [], []
    for start in range(max(len(tokens) - n + 1, 0)):
        feature = tokens[start:start + n]
        label_index = start + n
        label = tokens[label_index] if label_index < len(tokens) else None
        features.append(feature)
        labels.append(label)

    return vocab, (features, labels)

vocab, (features, labels) = preprocess_text("The time machine", 2)
print("vocab =", vocab)
print("features =", features)
print("labels =", labels)

vocab2, (features2, labels2) = preprocess_text("The Time, the Machine! The future.", 2)
print("\nsecond vocab =", vocab2)
print("second features =", features2)
print("second labels =", labels2)

vocab = {'the': 0, 'time': 1, 'machine': 2}
features = [['the', 'time'], ['time', 'machine']]
labels = ['machine', None]

second vocab = {'the': 0, 'time': 1, 'machine': 2, 'future': 3}
second features = [['the', 'time'], ['time', 'the'], ['the', 'machine'], ['machine', 'the'], ['the', 'future']]
second labels = ['the', 'machine', 'the', 'future', None]


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN 为

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t, \qquad o_t=W_{oh}h_t
$$

平方损失为

$$
L=\frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2
$$

令

$$
e_t=o_t-y_t, \qquad \delta_t=\frac{\partial L}{\partial h_t}
$$

通过时间反向传播有递推式

$$
\delta_t=W_{oh}^{T}e_t+W_{hh}^{T}\delta_{t+1}, \qquad \delta_{T+1}=0
$$

展开可得

$$
\delta_t=\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}e_k
$$

因此损失对 $W_{hh}$ 的梯度为

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\delta_t h_{t-1}^{T}
=\sum_{t=1}^{T}\left[\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}(o_k-y_k)\right]h_{t-1}^{T}
$$

梯度消失或爆炸主要由反复相乘的 $(W_{hh}^{T})^{k-t}$ 决定。若 $W_{hh}$ 的谱半径或最大奇异值小于 1，长时间步梯度会趋向于 0；若大于 1，梯度可能指数级放大。

### 3.2 编程题

下面实现一个带 `tanh` 激活函数的 RNN 单元前向传播和单步反向传播。

In [3]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    da_t = dh_next * (1 - h_t ** 2)
    dx_t = da_t @ W_hx.T
    dh_prev = da_t @ W_hh.T
    dW_hx = x_t.T @ da_t
    dW_hh = h_prev.T @ da_t
    db_h = da_t.sum(axis=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

np.random.seed(42)
batch_size, input_size, hidden_size = 2, 3, 4
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)
dh_next = np.random.randn(batch_size, hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)
print("h_t =\n", np.round(h_t, 4))

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)
h_t =
 [[-0.9995  0.8825 -0.9966 -0.617 ]
 [ 0.7649 -0.9758 -0.9996 -0.3702]]


## 4 高级循环神经网络

### 4.1 理论计算题

对于一个深度双向 RNN，每层有正向和反向两个 RNN。假设每个 RNN 单元使用一个输入权重、一个隐藏状态权重和一个偏置。

第 1 层每个方向的参数量为

$$
DH+H^2+H
$$

两个方向合计

$$
2(DH+H^2+H)
$$

从第 2 层开始，每个方向的输入维度为上一层双向输出的拼接维度 $2H$，所以每层两个方向的参数量为

$$
2(2H\cdot H+H^2+H)=6H^2+2H
$$

最后输出层从 $2H$ 映射到 $O$，参数量为

$$
2HO+O
$$

因此总参数量为

$$
\boxed{2(DH+H^2+H)+(L-1)(6H^2+2H)+2HO+O}
$$

其中忽略嵌入层和输出层之前的投影。

### 4.2 编程题

使用 `torch.nn.RNN` 实现双向 RNN 编码器，返回每个时间步的双向隐藏状态拼接结果和最终序列表示。

In [4]:
import torch
from torch import nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        final_forward = h_n[-2]
        final_backward = h_n[-1]
        final_hidden = torch.cat([final_forward, final_backward], dim=1)
        return outputs, final_hidden

torch.manual_seed(42)
seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 6
X = torch.randn(seq_len, batch, input_dim)
encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim)
outputs, final_hidden = encoder(X)

print("outputs shape:", tuple(outputs.shape))
print("final_hidden shape:", tuple(final_hidden.shape))

outputs shape: (5, 3, 12)
final_hidden shape: (3, 12)


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 的目标是给定中心词 $w_c$ 预测上下文词 $w_o$。使用负采样时，对每个正样本 $(w_c,w_o)$，从噪声分布 $P_n(w)$ 中采样 $K$ 个负样本 $n_1,\ldots,n_K$。常见做法是令

$$
P_n(w)=\frac{U(w)^{3/4}}{\sum_{w'\in V}U(w')^{3/4}}
$$

其中 $U(w)$ 是词在语料中的 unigram 频率。

设中心词向量为 $v_c$，上下文词输出向量为 $u_o$，负样本输出向量为 $u_{n_k}$，最大化的对数似然为

$$
\log \sigma(u_o^T v_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{T}v_c)
$$

对应要最小化的损失函数为

$$
\boxed{J=-\log \sigma(u_o^T v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{T}v_c)}
$$

负样本通常按 $P_n(w)$ 独立采样，实际实现中一般会避免把当前正样本上下文词当作负样本。

### 5.2 编程题

实现 CBOW 模型的前向传播和完整 softmax 交叉熵损失。

In [5]:
import torch
import torch.nn.functional as F

def cbow_loss(context_indices, target_indices, W, W_out):
    context_vectors = W[context_indices]
    hidden = context_vectors.mean(dim=1)
    logits = hidden @ W_out
    loss = F.cross_entropy(logits, target_indices)
    probabilities = F.softmax(logits, dim=1)
    return loss, probabilities

torch.manual_seed(42)
V, d, context_size, batch_size = 8, 5, 4, 3
context_indices = torch.tensor([
    [0, 1, 2, 3],
    [1, 2, 3, 4],
    [2, 3, 4, 5],
])
target_indices = torch.tensor([4, 5, 6])
W = torch.randn(V, d)
W_out = torch.randn(d, V)

loss, probabilities = cbow_loss(context_indices, target_indices, W, W_out)
print("loss =", round(loss.item(), 6))
print("probabilities shape:", tuple(probabilities.shape))
print("row sums:", probabilities.sum(dim=1))

loss = 2.029639
probabilities shape: (3, 8)
row sums: tensor([1.0000, 1.0000, 1.0000])


## 6 注意力机制

### 6.1 理论计算题

题目只给出 $Q\in\mathbb{R}^{2\times 4}$、$K\in\mathbb{R}^{3\times 4}$、$V\in\mathbb{R}^{3\times 5}$ 的形状，没有给出具体矩阵元素，因此这里写出符号计算过程。

因为 $d_k=4$，所以 $\sqrt{d_k}=2$。先计算得分矩阵

$$
S=\frac{QK^T}{\sqrt{d_k}}=\frac{QK^T}{2}\in\mathbb{R}^{2\times 3}
$$

若 $q_i$ 表示 $Q$ 的第 $i$ 行，$k_j$ 表示 $K$ 的第 $j$ 行，则

$$
S=\begin{bmatrix}
\frac{q_1\cdot k_1}{2} & \frac{q_1\cdot k_2}{2} & \frac{q_1\cdot k_3}{2}\\
\frac{q_2\cdot k_1}{2} & \frac{q_2\cdot k_2}{2} & \frac{q_2\cdot k_3}{2}
\end{bmatrix}
$$

然后对每一行做 softmax：

$$
A=softmax(S)\in\mathbb{R}^{2\times 3},\qquad
\alpha_{ij}=\frac{\exp(S_{ij})}{\sum_{m=1}^{3}\exp(S_{im})}
$$

最后加权求和：

$$
O=AV\in\mathbb{R}^{2\times 5}
$$

也就是

$$
O_i=\sum_{j=1}^{3}\alpha_{ij}v_j,\qquad i=1,2
$$

其中 $v_j$ 是 $V$ 的第 $j$ 行。

In [6]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return scores, weights, output

torch.manual_seed(42)
Q = torch.randn(2, 4)
K = torch.randn(3, 4)
V = torch.randn(3, 5)
scores, weights, output = scaled_dot_product_attention(Q, K, V)

print("scores shape:", tuple(scores.shape))
print("weights shape:", tuple(weights.shape))
print("output shape:", tuple(output.shape))
print("scores =\n", torch.round(scores, decimals=4))
print("attention weights row sums:", weights.sum(dim=1))

scores shape: (2, 3)
weights shape: (2, 3)
output shape: (2, 5)
scores =
 tensor([[ 0.2509,  0.0725,  0.0990],
        [ 0.0483, -1.8634, -1.4245]])
attention weights row sums: tensor([1.0000, 1.0000])


### 6.2 编程题

实现 `num_heads=2`, `d_model=4` 的多头注意力前向传播。

In [7]:
import math
import torch
from torch import nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, x):
        seq_len, batch, _ = x.shape
        x = x.view(seq_len, batch, self.num_heads, self.d_k)
        return x.permute(1, 2, 0, 3)

    def _combine_heads(self, x):
        batch, heads, seq_len, d_k = x.shape
        x = x.permute(2, 0, 1, 3).contiguous()
        return x.view(seq_len, batch, heads * d_k)

    def forward(self, X):
        Q = self._split_heads(self.W_q(X))
        K = self._split_heads(self.W_k(X))
        V = self._split_heads(self.W_v(X))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        head_outputs = weights @ V
        concat = self._combine_heads(head_outputs)
        return self.W_o(concat)

torch.manual_seed(42)
seq_len, batch, d_model = 6, 2, 4
X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(num_heads=2, d_model=4)
Y = mha(X)

print("input shape:", tuple(X.shape))
print("output shape:", tuple(Y.shape))

input shape: (6, 2, 4)
output shape: (6, 2, 4)
